# Gene Set Enrichment Analysis for all Perturbations against GO Databases

## Import

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys 
import time
import pandas as pd 
import numpy as np 
import scanpy as sc 
import muon as mu
import idea

sys.path.append('../utils')

import signature_heatmaps as signature_heatmaps
import factor_labels as factor_labels


## Load Data

In [ ]:
data_dir = "<path to processed data>"

In [ ]:
cite_6tf_path = os.path.join(data_dir, "cite_6tf_cleaned_revisions.h5mu")
cite_imgl_path = os.path.join(data_dir, "cite_imgl_cleaned_revisions.h5mu")
merged_6tf_path = os.path.join(data_dir, "adata_revisions_merged_6tf.h5ad")

In [ ]:
mdata_dict = {}
mdata_dict['cite_6tf'] = mu.read_h5mu(cite_6tf_path)
mdata_dict['cite_imgl'] = mu.read_h5mu(cite_imgl_path)

adata_dict = {}
adata_dict['merged_6tf'] = sc.read_h5ad(merged_6tf_path)
adata_dict['cite_6tf'] = mdata_dict['cite_6tf'].mod['rna'].copy()
adata_dict['cite_imgl'] = mdata_dict['cite_imgl'].mod['rna'].copy()

In [ ]:
signature_cols_ordered = ['homeostatic_score_ucell',
 'interferon_score_ucell',
 'chemokine_score_ucell',
 'antigen_presenting_score_ucell',
 'dam_score_ucell',
 'lipid_dam_score_ucell']

## Masking for analysis
to exclude ntc_g5 + foxk1_g2 + mixscale_cutoff >= 0

In [ ]:
guides_to_exclude = ['FOXK1_g2', 'non-targeting_g5']

adata_6tf_clean = adata_dict['merged_6tf'][~adata_dict['merged_6tf'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
adata_imgl_clean = adata_dict['cite_imgl'][~adata_dict['cite_imgl'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
adata_6tf_clean.shape, adata_imgl_clean.shape

In [ ]:
print(mdata_dict['cite_6tf'].shape, mdata_dict['cite_imgl'].shape)
mdata_6tf_clean = mdata_dict['cite_6tf'][~mdata_dict['cite_6tf'].mod['rna'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
mdata_imgl_clean = mdata_dict['cite_imgl'][~mdata_dict['cite_imgl'].mod['rna'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
mdata_6tf_clean.shape, mdata_imgl_clean.shape

In [ ]:
mixscale_col = "mixscale_score"

In [ ]:
adata_6tf_masked = adata_6tf_clean[adata_6tf_clean.obs[mixscale_col] >= 0].copy()
adata_imgl_masked = adata_imgl_clean[adata_imgl_clean.obs[mixscale_col] >= 0].copy()
adata_6tf_masked.shape, adata_imgl_masked.shape

In [ ]:
adata_masked_dict = {}
adata_masked_dict['iTF'] = adata_6tf_masked
adata_masked_dict['iMG'] = adata_imgl_masked

# Genes to incude

In [ ]:
genes_to_include = ['DNMT1', 'IRF9', 'STAT2', 'SMAD3', 'PRDM1', 'ZNF532']
itf_genes = ['DNMT1', 'IRF9', 'STAT2', 'SMAD3']
img_genes = ['PRDM1', 'ZNF532']

In [ ]:
all_genes = sorted(adata_imgl_masked.obs.perturbed_gene.unique().tolist())

# Load in Differentially Expressed Genes (DEG) Dictionary

In [ ]:
def get_all_kd_DE(deg_dir, 
                  genes_to_include, 
                  model_names_list, 
                  logfc_threshold = 0, 
                  adj_pval_threshold = 0.05, 
                  type = "rna") -> (dict, dict):
    deg_dict = {}
    sig_deg_dict = {}
    genes = genes_to_include
    for model in model_names_list:
        sig_deg_dict[model] = {}
        deg_dict[model] = {}
        for gene in genes:
            if gene == "NTC":
                continue
            if (type == "rna") & (model == "merged_6tf") :
                path = deg_dir + f"/{model}_mixscale_degs_all_{type}_{gene}.csv"
            elif (type == "rna") & (model == "cite_imgl"):
                path = deg_dir + f"/{model}_mixscale_degs_all_{type}_{gene}.csv"
            else:
                path = deg_dir + f"/{model}_mixscale_degs_all_{type}_{gene}.csv"
            if os.path.exists(path):
                deg_dict[model][gene] = pd.read_csv(path, index_col = 0)
                deg_dict[model][gene]['gene_ID'] = deg_dict[model][gene].index
                sig_deg_dict[model][gene] = {}
                sig_deg_dict[model][gene]['df'] = deg_dict[model][gene][deg_dict[model][gene]['adj_p_weight'] <= adj_pval_threshold].copy()
                sig_deg_dict[model][gene]['negative'] = sig_deg_dict[model][gene]['df'][sig_deg_dict[model][gene]['df']['log2FC'] <= logfc_threshold].copy()
                sig_deg_dict[model][gene]['positive'] = sig_deg_dict[model][gene]['df'][sig_deg_dict[model][gene]['df']['log2FC'] > logfc_threshold].copy()
                
    return deg_dict, sig_deg_dict

In [ ]:
deg_dir = "<path to Mixscale DEG directory>"
kds_to_include = set(adata_6tf_clean.obs['perturbed_gene'])

In [ ]:
kds_to_include.remove("NTC")

In [ ]:
deg_rna_dict, sig_deg_rna_dict = get_all_kd_DE(deg_dir, 
                                               kds_to_include, 
                                               ['merged_6tf', 'cite_imgl'], 
                                               logfc_threshold = 0, 
                                               adj_pval_threshold = 0.05, 
                                               type = "rna")

In [ ]:
dap_prot_dict, sig_dap_prot_dict = get_all_kd_DE(deg_dir, 
                                               kds_to_include, 
                                               ['merged_6tf', 'cite_imgl'], 
                                               logfc_threshold = 0, 
                                               adj_pval_threshold = 0.05, 
                                               type = "prot")

In [ ]:
def format_deg_dict_to_df(deg_dict, model, type="rna", savetable=None) -> pd.DataFrame():
    deg_df = pd.concat(
        {key: df for key, df in deg_dict[model].items()}, names=['knockdown']
        ).reset_index(level=0).reset_index(drop=True)
    
    deg_df = deg_df[['knockdown', 'gene_ID', 'log2FC', 'beta_weight', 'p_weight', 'DE_method','adj_p_weight']]
    gene_col = "gene_name" if type == "rna" else "protein_name"
    deg_df = deg_df.rename(columns={'gene_ID':gene_col, 'log2FC': 'log2fc',
                           'p_weight':'pvalue',
                           'DE_method':'de_method',
                           'adj_p_weight':'adj_pvalue_BH'})
    deg_df = deg_df.sort_values(by="knockdown").reset_index(drop=True)
    deg_df['significant'] = np.where(deg_df['adj_pvalue_BH'] < 0.05, True, False)
    if savetable:
        deg_df.to_csv(savetable)
    return deg_df

In [ ]:
format_deg_dict_to_df(deg_rna_dict, "merged_6tf", type="rna",
                      savetable=os.path.join(data_dir, "deg_rna_itf_all.csv"))

In [ ]:
format_deg_dict_to_df(deg_rna_dict, "cite_imgl", type="rna",
                      savetable=os.path.join(data_dir, "deg_rna_img_all.csv"))

In [ ]:
format_deg_dict_to_df(dap_prot_dict, "cite_imgl", type="prot",
                      savetable=os.path.join(data_dir, "dap_prot_img_all.csv"))

In [ ]:
format_deg_dict_to_df(dap_prot_dict, "merged_6tf", type="prot",
                      savetable=os.path.join(data_dir, "dap_prot_itf_all.csv"))

# GSEA analysis using IDEA (backend = EnrichR) for RNA DEGs

In [ ]:
go_name_dict = {'BP': "GO_Biological_Process_2023",
                "MF": "GO_Molecular_Function_2023",
                "CC": "GO_Cellular_Component_2023"}

#### iMG

In [ ]:
dfs = []
for g in img_genes:
    pos_list = sig_deg_rna_dict['cite_imgl'][g]['positive']['gene_ID'].tolist()
    neg_list = sig_deg_rna_dict['cite_imgl'][g]['negative']['gene_ID'].tolist()
    print(g, f"num_sig_pos: {len(pos_list)}", f"num_sig_neg: {len(neg_list)}")
    gene_dfs = []
    for lib in ['BP', 'MF', 'CC']:
        for dir, dir_list in zip(['positive', 'negative'], [pos_list, neg_list]):
            if len(dir_list) > 100:
                dir_list = dir_list[:100]
            df = idea.run_gsea(dir_list, lib)
            df['library'] = lib
            df['knockdown'] = g
            df['deg_direction'] = dir
            gene_dfs.append(df)
            time.sleep(2.5)
    dfs.append(pd.concat(gene_dfs, ignore_index=True))
gsea_df_iMG = pd.concat(dfs, ignore_index=True)
gsea_df_iMG['model'] = 'iMG'

#### Validation

In [ ]:
for g in img_genes:
    print(g)
    print(gsea_df_iMG[gsea_df_iMG['knockdown'] == g]['deg_direction'].value_counts())
    print(gsea_df_iMG[gsea_df_iMG['knockdown'] == g]['library'].value_counts())

In [ ]:
gsea_df_iMG = gsea_df_iMG.drop(['rank', 'old_pvalue', 'old_adj_pvalue'], axis=1)
gsea_df_iMG = gsea_df_iMG[['model','knockdown','deg_direction', 'library', 'term_name',
                           'pvalue', "zscore", "combined_score", "overlapping_genes", "adj_pvalue"]]
gsea_df_iMG['library'] = gsea_df_iMG['library'].map(go_name_dict)

In [ ]:
gsea_df_iMG.to_csv(os.path.join(data_dir, "gsea_rna_degs_iMG.csv"), index=False)

#### iTF

In [ ]:
dfs = []
for g in itf_genes:
    pos_list = sig_deg_rna_dict['merged_6tf'][g]['positive']['gene_ID'].tolist()
    neg_list = sig_deg_rna_dict['merged_6tf'][g]['negative']['gene_ID'].tolist()
    print(g, f"num_sig_pos: {len(pos_list)}", f"num_sig_neg: {len(neg_list)}")
    gene_dfs = []
    for dir, dir_list in zip(['positive', 'negative'], [pos_list, neg_list]):
        if len(dir_list) > 100:
            dir_list = dir_list[:100]
        if len(dir_list) <= 1:
            continue
        for lib in ['BP', 'MF', 'CC']:
            df = idea.run_gsea(dir_list, lib)
            df['library'] = lib
            df['knockdown'] = g
            df['deg_direction'] = dir
            gene_dfs.append(df)
            time.sleep(2.5)
    dfs.append(pd.concat(gene_dfs, ignore_index=True))
gsea_df_iTF= pd.concat(dfs, ignore_index=True)
gsea_df_iTF['model'] = 'iTF-MG'

#### Validation

In [ ]:
for g in itf_genes:
    print(g)
    print(gsea_df_iTF[gsea_df_iTF['knockdown'] == g]['deg_direction'].value_counts())
    print(gsea_df_iTF[gsea_df_iTF['knockdown'] == g]['library'].value_counts())

In [ ]:
gsea_df_iTF = gsea_df_iTF.drop(['rank', 'old_pvalue', 'old_adj_pvalue'], axis=1)
gsea_df_iTF = gsea_df_iTF[['model','knockdown','deg_direction', 'library', 'term_name',
                           'pvalue', "zscore", "combined_score", "overlapping_genes", "adj_pvalue"]]
gsea_df_iTF['library'] = gsea_df_iTF['library'].map(go_name_dict)

In [ ]:
gsea_df_iTF.to_csv(os.path.join(data_dir, "gsea_rna_degs_iTF-MG.csv"), index=False)

## Combine iTF and iMG GSEA into one dataframe

In [ ]:
gsea_df_all = pd.concat([gsea_df_iMG, gsea_df_iTF])

In [ ]:
gsea_df_all.to_csv(os.path.join(data_dir, "gsea_rna_degs_combined.csv"), index=False)